# LLM Direct Hypotheses Runner

**Goal:** a comparator for CLIP-H. An LLM writes clinical **hypotheses** straight from labeled
train profiles (no SAE, no curation beyond exact dedup); the patients are then **annotated**
against the hypotheses and a logistic regression fit on train is **evaluated** on held-out test
patients. Annotation and the held-out evaluation follow CLIP-H Phase 3A, so the two methods
differ in where the hypotheses come from.

### Method in one paragraph
Four seeded, class-balanced batches of 20 labeled train profiles are shown to the generator,
which proposes 20 hypotheses per batch ("Patient has/is/was X", observable in the profile, not
the outcome itself). The pool of 80 is deduplicated on exact text, taking one hypothesis from
each batch in turn, and the first 15 form the library. Every train and test patient is
annotated against the library with the CLIP-H annotation code and prompt (one call per patient
for all hypotheses), a cross-validated logistic regression is fit on the train annotations
(5,000 patients; 1,000 for mortality and discharge, as in the reported runs), and the test
patients are scored.

### Map of this notebook
Run the cells top to bottom. Every step first looks for its saved file; if the file is there the step loads it and makes no call. `FORCE_RERUN` (Settings) redoes a step.

| # | Cell | What it does | Reads | If its file exists | If not | Writes | LLM calls |
|---|------|--------------|-------|--------------------|--------|--------|-----------|
| 1 | Settings | Every setting of the run | | | | | none |
| 2 | Setup | Finds this folder and the task's data, makes result folders, API key, call cache, usage log | `code/`, `prompts/` | | | `caches/usage_log.jsonl`, `caches/llm_call_cache.jsonl` (as calls happen) | none |
| 3 | Phase 0 | Loads profiles and labels | `train.jsonl`, `test.jsonl` | loads them | builds them from `*_profile.jsonl` + `*_labels.csv` | `run_manifest.json` | none |
| 4 | Phase G1 | Labeled batches, one generation call per batch | `phaseG_generate/generation_pool.json` | loads it (same settings only) | generates | that file | `GEN_MODEL` |
| 5 | Phase G2 | Exact-text dedup, one per batch in turn, first `K_HYPOTHESES` | the pool | | | `phaseG_generate/generated_hypotheses.csv` | none |
| 6 | Phase 3A | Annotates TEST and TRAIN, fits the prediction model, 95% intervals | (labels from the caches) | | annotates what is missing | `phase3_prediction/annotation_matrix.npz`, `heldout_metrics.json`, `hypothesis_lr_coefs.csv`, `model_metrics.csv` | `ANNOT_MODEL` |
| 7 | Phase 3B | Checks that every expected file exists, prints the summary and the cost | all of the above | | stops and lists what is missing | `run_config.json` | none |

**Calls are never paid twice.** Every reply is saved in `caches/llm_call_cache.jsonl` and every label in a `caches/*_annotation_cache.json` file as soon as it arrives, so an interrupted cell can simply be run again. A label that could not be obtained stops Phase 3A; running the cell again asks for it.

**The two runner notebooks** (`direct_hypotheses_gpt.ipynb`, `direct_hypotheses_deepseek.ipynb`) differ in one line: `MODEL_PROFILE` in Settings. `TASK` (Settings) picks one of the five tasks.

### Where things are
- Input data: `data/<task>/` at the repository root (`TASK_DIR`, setting `TASK_DATA_REL`): `*_profile.jsonl`, `*_labels.csv` per split
- Code: `code/` (the files copied from CLIP-H are the same as in `clip_h/readmission/code/`; `common/annotation.py` adds `use_prompt_file`) · Prompts: `prompts/` (`prompts/README.md` lists the sections)
- Results: `results/<RUN_NAME>/<task>_<arm>/` with one folder per phase and `caches/`. A new `RUN_NAME` starts a new, empty run folder; it reads the saved replies and labels of the other runs (`REUSE_CACHES_FROM`)
- The patient data is not included (MIMIC-IV needs credentialed access).

## 1. Settings

In [ ]:
import os

# ---- Task: TASK picks one row of _TASKS -------------------------------------------------------
TASK = 'readmission'        # 'readmission' | 'hosp_los3' | 'icu_los4' | 'mortality' | 'discharge' (the folder name in data/)
_TASKS = {   # class names in printed lines, train patients the model is fit on, generation wording
    'readmission': dict(pos_neg=('readmitted', 'not readmitted'), gate_train_n=5000,
         outcome='the patient had an unplanned readmission to the hospital within 30 days of discharge (Yes) or not (No)',
         extra_rules=''),
    'hosp_los3': dict(pos_neg=('long stay', 'not long stay'), gate_train_n=5000,
         outcome='the hospital stay lasted MORE than 3 days (Yes) or 3 days or less (No)',
         extra_rules='- Do not propose hypotheses about the length of the hospital stay itself.'),
    'icu_los4': dict(pos_neg=('long ICU stay', 'not long ICU stay'), gate_train_n=5000,
         outcome='the ICU stay lasted MORE than 4 days (Yes) or 4 days or less (No)',
         extra_rules='- Every patient in this dataset is an ICU patient: do NOT propose hypotheses about ICU admission itself.\n- Do not propose hypotheses about the length of any stay (hospital or ICU).'),
    'mortality': dict(pos_neg=('died', 'survived'), gate_train_n=1000,
         outcome='the patient died during the indexed hospitalization (Yes) or survived to discharge (No)',
         extra_rules=''),
    'discharge': dict(pos_neg=('home', 'not home'), gate_train_n=1000,
         outcome='the patient was discharged to a private home without services (Yes) or anywhere else (No)',
         extra_rules='- Do not propose hypotheses about the discharge destination itself.'),
}
_T = _TASKS[TASK]
TASK_NAME = TASK            # the arm's results go to results/<RUN_NAME>/<TASK_NAME>_<MODEL_PROFILE>/
TASK_DATA_REL = f'../data/{TASK}'   # <repo>/data/<task>/: train/test _profile.jsonl and _labels.csv, relative to this folder
POS_NAME, NEG_NAME = _T['pos_neg']
ANNOTATION_PROMPT = f'annotation_{TASK}'   # prompts/<file>.txt, the same text as the task's CLIP-H annotation.txt

# ---- Run ---------------------------------------------------------------------------------
MODULE_FOLDER = None        # this folder (the one holding code/ and prompts/). None = found from the working
                            # directory; set a path (relative or absolute) when the notebook is started elsewhere
RUN_NAME = 'direct_full'    # every result of this run goes to results/<RUN_NAME>/. A new name = a new, empty run folder
REUSE_CACHES_FROM = 'all'   # other runs whose saved replies and labels this run may read (never written to), same arm only.
                            # 'all' = every other run folder in results/; a list of run names = only those; [] = none.
SEED = 42                   # generation batches, LR cross-validation, the tie-break at the F1 cut, the intervals

# ---- Models: MODEL_PROFILE picks the arm ---------------------------------------------------
MODEL_PROFILE = 'deepseek'      # 'gpt' | 'deepseek'
_MODEL_PROFILES = {
    'gpt':      dict(gen='gpt-4.1', annot='gpt-4.1-mini', base_url=None,
                     annotate_style='checklist', annotate_workers=16, gen_max_tokens=8000, timeout=300.0),
    'deepseek': dict(gen='deepseek-v4-pro', annot='deepseek-v4-flash', base_url='https://api.deepseek.com',
                     annotate_style='labels', annotate_workers=8,
                     gen_max_tokens=8000,      # thinking off in every call, as in CLIP-H (code/common/llm_client.py)
                     timeout=600.0),           # not used by DeepSeek models: their calls are streamed
}
_PROFILE = _MODEL_PROFILES[MODEL_PROFILE]
GEN_MODEL = _PROFILE['gen']          # Phase G1 generation
ANNOT_MODEL = _PROFILE['annot']      # Phase 3A annotation
LLM_TIMEOUT = _PROFILE['timeout']
ANNOTATE_STYLE = _PROFILE['annotate_style']       # reply format of the annotation call: 'checklist' | 'labels' (prompts/annotation_<task>.txt)
ANNOTATE_WORKERS = _PROFILE['annotate_workers']   # parallel annotation calls

# ---- Data (Phase 0) ------------------------------------------------------------------------
N_PATIENTS_TRAIN = 5000   # label-stratified subsample, seeded with SEED (None = all)
N_PATIENTS_TEST = 1000

# ---- Phase G1 and G2: generation -----------------------------------------------------------
GEN_OUTCOME = _T['outcome']
GEN_EXTRA_RULES = _T['extra_rules']   # task-specific rules, one '- ...' line each
GEN_BATCHES = 4           # generation calls
GEN_BATCH_SIZE = 20       # labeled profiles per call, half Yes / half No
GEN_PER_BATCH = 20        # hypotheses asked for per call
GEN_TEMPERATURE = 0.7     # diversity across batches
MIN_PER_BATCH = 10        # a reply with fewer hypotheses is asked again
MAX_GEN_ATTEMPTS = 6
K_HYPOTHESES = 15         # library size

# ---- Phase 3A ------------------------------------------------------------------------------
GATE_TRAIN_N = _T['gate_train_n']   # training profiles annotated to fit the prediction model (None = all)

# ---- What to redo: False reuses the saved file of that step --------------------------------
FORCE_RERUN = {
    'handoff_jsonl': False,   # Phase 0: train.jsonl / test.jsonl
    'generation': False,      # Phase G1 (LLM calls)
}

print('Settings loaded. Task:', TASK, '| profile:', MODEL_PROFILE, '| generate / annotate:', GEN_MODEL, '/', ANNOT_MODEL, '| K:', K_HYPOTHESES)

## 2. Setup

In [ ]:
import sys, json, getpass, hashlib
from datetime import datetime
from functools import partial
from pathlib import Path
import numpy as np
import pandas as pd

# ---- 1. Find the module folder: the one holding code/code_paths.py ------------------------
def is_module_root(path):
    return (Path(path) / 'code' / 'code_paths.py').exists()


_candidates = [Path(MODULE_FOLDER).expanduser()] if MODULE_FOLDER else [Path.cwd(), *Path.cwd().resolve().parents]
MODULE_FOLDER = next((p.resolve() for p in _candidates if is_module_root(p)), None)
if MODULE_FOLDER is None:
    raise FileNotFoundError('Module folder not found. Start the notebook from inside direct_llm_hypotheses/, '
                            'or set MODULE_FOLDER in Settings to its path.')
TASK_DIR = (MODULE_FOLDER / TASK_DATA_REL).resolve()
_missing = [f for f in ('train_profile.jsonl', 'test_profile.jsonl') if not (TASK_DIR / f).exists()]
if _missing:
    raise FileNotFoundError(f'{_missing} not found in {TASK_DIR}. Write data/{TASK}/ first (see the README at the '
                            'repository root), or set TASK_DATA_REL to your own folder.')

# ---- 2. Result folders ---------------------------------------------------------------------
RUN_DIR = MODULE_FOLDER / 'results' / RUN_NAME
RESULTS_DIR = RUN_DIR / f'{TASK_NAME}_{MODEL_PROFILE}'
PG_DIR = RESULTS_DIR / 'phaseG_generate'
P3_DIR = RESULTS_DIR / 'phase3_prediction'
CACHE_DIR = RESULTS_DIR / 'caches'
for _dir in (PG_DIR, P3_DIR, CACHE_DIR):
    _dir.mkdir(parents=True, exist_ok=True)
print('Module folder:', MODULE_FOLDER, '\nTask data:    ', TASK_DIR, '\nResults:      ', RESULTS_DIR)

# ---- 3. Code: code/ holds one folder per step; code_paths makes them importable -------------
CODE_DIR = MODULE_FOLDER / 'code'
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
import code_paths

# ---- 4. API key: asked for once, kept only in this session's environment --------------------
_base_url = _PROFILE['base_url']
if _base_url:   # DeepSeek: an OpenAI-compatible endpoint
    os.environ['DEEPSEEK_API_KEY'] = os.environ.get('DEEPSEEK_API_KEY') or getpass.getpass('DeepSeek API key: ')
    os.environ['OPENAI_KEY_SAE'] = os.environ['DEEPSEEK_API_KEY']
    os.environ['OPENAI_BASE_URL_SAE'] = _base_url
else:
    if 'OPENAI_KEY_SAE' not in os.environ:
        os.environ['OPENAI_KEY_SAE'] = os.environ['OPENAI_API_KEY'] = os.environ.get('OPENAI_API_KEY') or getpass.getpass('OpenAI API key: ')
    os.environ['OPENAI_BASE_URL_SAE'] = 'https://api.openai.com/v1'
print('Provider:', os.environ['OPENAI_BASE_URL_SAE'])

# ---- 5. LLM client, call cache, usage log (code/common/llm_client.py) -----------------------
import llm_client
from llm_client import set_default_timeout, install_usage_meter, install_call_cache, usage_report
from prompts import load_prompt   # reads prompts/<file>.txt
from clip_h_data import read_json, write_json, json_safe
import phase_tables as PT

set_default_timeout(LLM_TIMEOUT)
install_usage_meter(CACHE_DIR / 'usage_log.jsonl')
if REUSE_CACHES_FROM == 'all':
    _reuse_names = sorted(p.name for p in (MODULE_FOLDER / 'results').iterdir()
                          if p.name != RUN_NAME and (p / f'{TASK_NAME}_{MODEL_PROFILE}' / 'caches').is_dir())
else:
    _reuse_names = list(REUSE_CACHES_FROM)
_reuse_dirs = [MODULE_FOLDER / 'results' / name / f'{TASK_NAME}_{MODEL_PROFILE}' / 'caches' for name in _reuse_names]
_reuse_dirs = [d for d in _reuse_dirs if d.exists() and d != CACHE_DIR]
print('Caches of other runs read by this run:', [d.parent.parent.name for d in _reuse_dirs] or 'none')
install_call_cache(CACHE_DIR / 'llm_call_cache.jsonl',
                   reuse=[d / 'llm_call_cache.jsonl' for d in _reuse_dirs if (d / 'llm_call_cache.jsonl').exists()])
pd.set_option('display.max_colwidth', 90)
pd.set_option('display.width', 220)

# ---- 6. Annotation, used by Phase 3A (code/common/annotation.py, CLIP-H's file) -------------
from annotation import build_annotation_matrix as _build_annotation_matrix, set_reuse_cache_dirs, failed_labels, use_prompt_file
use_prompt_file(ANNOTATION_PROMPT)   # the task's prompts/annotation_<task>.txt
set_reuse_cache_dirs(_reuse_dirs)
build_annotation_matrix = partial(_build_annotation_matrix, mode='binary', model=ANNOT_MODEL,
                                  n_workers=ANNOTATE_WORKERS, style=ANNOTATE_STYLE)
print(f'Setup done. Timeout {LLM_TIMEOUT:.0f}s, {ANNOTATE_WORKERS} parallel annotation calls, reply style {ANNOTATE_STYLE!r}.')

## 3. Phase 0: load the data

In [ ]:
from clip_h_data import prepare_split, report_splits, stratified_subsample

force = FORCE_RERUN['handoff_jsonl']
train_profiles, y_train, ids_train, train_jsonl = prepare_split('train', TASK_DIR, force=force, limit=None)
test_profiles,  y_test,  ids_test,  test_jsonl  = prepare_split('test',  TASK_DIR, force=force, limit=None)
train_profiles, y_train, ids_train = stratified_subsample(train_profiles, y_train, ids_train, N_PATIENTS_TRAIN, SEED)
test_profiles,  y_test,  ids_test  = stratified_subsample(test_profiles,  y_test,  ids_test,  N_PATIENTS_TEST,  SEED)
y_train, y_test = np.asarray(y_train, dtype=int), np.asarray(y_test, dtype=int)

report_splits({'Train': y_train, 'Test': y_test}, pos_name=POS_NAME, neg_name=NEG_NAME)
write_json(RESULTS_DIR / 'run_manifest.json', {
    'task': TASK_NAME, 'run_name': RUN_NAME, 'train_jsonl': str(train_jsonl), 'test_jsonl': str(test_jsonl),
    'n_train': len(train_profiles), 'n_test': len(test_profiles),
    'train_pos_rate': float(y_train.mean()), 'test_pos_rate': float(y_test.mean()),
    'model_profile': MODEL_PROFILE, 'gen_model': GEN_MODEL, 'annot_model': ANNOT_MODEL,
})
print('Wrote', RESULTS_DIR / 'run_manifest.json')

## 4. Phase G1: generate hypotheses from labeled batches

In [ ]:
from direct_generation import generation_batches, generation_prompt, generate_pool, saved_pool

pool_path = PG_DIR / 'generation_pool.json'
GEN_SETTINGS = dict(
    gen_model=GEN_MODEL.removesuffix('-think'),
    # DeepSeek's thinking mode, recorded so that a pool made in the other mode is never reused.
    **({'gen_thinking': GEN_MODEL.endswith('-think')} if 'deepseek' in GEN_MODEL else {}),
    gen_prompt_sha1=hashlib.sha1((load_prompt('generation', 'system') + '\n---\n'
                                             + load_prompt('generation', 'user')).encode('utf-8')).hexdigest(),
    gen_batches=GEN_BATCHES, gen_batch_size=GEN_BATCH_SIZE, gen_per_batch=GEN_PER_BATCH,
    gen_temperature=GEN_TEMPERATURE, seed=SEED, outcome=GEN_OUTCOME, extra_rules=GEN_EXTRA_RULES)

pool = None if FORCE_RERUN['generation'] else saved_pool(pool_path, RESULTS_DIR / 'run_config.json', GEN_SETTINGS)
if pool is not None:
    print('Loaded the saved pool (same settings):', [len(b) for b in pool])
else:
    batches = generation_batches(y_train, GEN_BATCHES, GEN_BATCH_SIZE, SEED)
    prompts = [generation_prompt(train_profiles, y_train, b, GEN_OUTCOME, GEN_EXTRA_RULES, GEN_PER_BATCH) for b in batches]
    pool = generate_pool(prompts, GEN_MODEL, GEN_TEMPERATURE, _PROFILE['gen_max_tokens'], MIN_PER_BATCH, MAX_GEN_ATTEMPTS,
                         timeout=LLM_TIMEOUT)
    write_json(pool_path, pool)
    write_json(RESULTS_DIR / 'run_config.json', {**read_json(RESULTS_DIR / 'run_config.json', {}), **GEN_SETTINGS})
print(f'Pool: {sum(map(len, pool))} hypotheses in {len(pool)} batches')

## 5. Phase G2: dedup and pick the library

In [ ]:
from direct_generation import pick_library

hypotheses = pick_library(pool, K_HYPOTHESES)
pd.DataFrame({'hypothesis': hypotheses}).to_csv(PG_DIR / 'generated_hypotheses.csv', index=False)
print(f'{sum(map(len, pool))} generated -> {len(hypotheses)} in the library')
for _i, _h in enumerate(hypotheses, 1):
    print(f'{_i:3d}. {_h}')

## 6. Phase 3A: annotate and predict

In [ ]:
from evaluation import bootstrap_intervals
from direct_fit import fit_and_score_heldout_cv, gate_train_rows

# 1. TEST and TRAIN annotations, each in its own label cache.
annotation_matrix = np.asarray(build_annotation_matrix(
    texts=test_profiles, concepts=hypotheses, cache_path=CACHE_DIR / 'binary_annotation_cache.json',
    label='Phase 3A test annotation'), dtype=float)
fit_rows = gate_train_rows(y_train, GATE_TRAIN_N, SEED)   # every train row when GATE_TRAIN_N covers them
n_eval_train, y_train_eval = len(fit_rows), y_train[fit_rows]
X_train_eval = np.asarray(build_annotation_matrix(
    texts=[train_profiles[i] for i in fit_rows], concepts=hypotheses, cache_path=CACHE_DIR / 'binary_train_annotation_cache.json',
    label='Phase 3A train annotation'), dtype=float)
_failed = {k: v for k, v in failed_labels(CACHE_DIR).items() if k.startswith('Phase 3A') and v['failed']}
if _failed:
    raise RuntimeError(f'Labels could not be obtained: {_failed}. Run this cell again to ask for them.')

# 2. The headline numbers: logistic regression fit on TRAIN annotations, scored on TEST.
heldout_metrics, test_score, model_coef = fit_and_score_heldout_cv(
    X_train_eval, y_train_eval, annotation_matrix, y_test, seed=SEED)
heldout_metrics['ci95'] = bootstrap_intervals(y_test, test_score, seed=SEED)   # 10,000 resamples of the test set, model not refit
(P3_DIR / 'heldout_metrics.json').write_text(json.dumps(heldout_metrics, indent=2), encoding='utf-8')
np.savez_compressed(P3_DIR / 'annotation_matrix.npz', annotation_matrix=annotation_matrix,
                    train_annotation_matrix=X_train_eval, hypotheses=np.array(hypotheses, dtype=object),
                    labels=y_test, train_labels=y_train_eval)
pd.DataFrame({'hypothesis': hypotheses, 'lr_coef': model_coef, 'train_prevalence': X_train_eval.mean(axis=0)}) \
    .sort_values('lr_coef', ascending=False).to_csv(P3_DIR / 'hypothesis_lr_coefs.csv', index=False)
pd.DataFrame([{'run_name': RUN_NAME, 'task': TASK_NAME, 'model': 'llm_direct_hypotheses', 'arm': MODEL_PROFILE,
               'n_train': int(n_eval_train), 'n_test': int(len(y_test)),
               **{k: heldout_metrics[k] for k in ('auroc', 'auprc', 'f1', 'precision', 'recall')}}]) \
    .to_csv(P3_DIR / 'model_metrics.csv', index=False)

print(f'Held-out result: logistic regression fit on {n_eval_train} train patients, scored on {len(y_test)} test patients, '
      f'{len(hypotheses)} hypotheses.')
PT.show(PT.metrics_table(heldout_metrics))

## 7. Phase 3B: summary

In [ ]:
expected = [RESULTS_DIR / 'run_manifest.json', PG_DIR / 'generation_pool.json', PG_DIR / 'generated_hypotheses.csv',
            P3_DIR / 'annotation_matrix.npz', P3_DIR / 'heldout_metrics.json', P3_DIR / 'hypothesis_lr_coefs.csv',
            P3_DIR / 'model_metrics.csv']
missing = [str(p) for p in expected if not p.exists()]
if missing:
    raise FileNotFoundError('Missing outputs:\n  ' + '\n  '.join(missing))

write_json(RESULTS_DIR / 'run_config.json', {
    **GEN_SETTINGS, 'task': TASK_NAME, 'run_name': RUN_NAME, 'model_profile': MODEL_PROFILE, 'annot_model': ANNOT_MODEL,
    'annotate_style': ANNOTATE_STYLE, 'n_train': len(train_profiles), 'n_test': len(test_profiles),
    'n_train_annotated': int(n_eval_train), 'k_hypotheses': K_HYPOTHESES,
    'annotation_prompt_sha1': hashlib.sha1((MODULE_FOLDER / 'prompts' / f'{ANNOTATION_PROMPT}.txt').read_bytes()).hexdigest(),
    'timestamp': datetime.now().isoformat(timespec='seconds')})
print(f"{TASK_NAME} | {MODEL_PROFILE} | {len(hypotheses)} hypotheses | AUROC {heldout_metrics['auroc']:.4f} "
      f"(95% {heldout_metrics['ci95']['auroc'][0]:.3f} to {heldout_metrics['ci95']['auroc'][1]:.3f}) | F1 {heldout_metrics['f1']:.3f}")
print('All outputs in', RESULTS_DIR)
usage_report(CACHE_DIR / 'usage_log.jsonl')